# Pruebas de mejora de `.geo` y `.sif`

Este notebook es para comparar variantes de geometr?a y mallado antes de meterlas al flujo gen?tico completo. La idea principal es probar dos familias:

- **`annular_rotor`**: el modelo actual, con dos imanes internos anulares que abrazan el rotor y dos anillos exteriores.
- **`central_pill`**: una variante inspirada en el esquema usado por Rub?n, con imanes internos s?lidos tipo pastilla dentro del impulsor, manteniendo los imanes exteriores y la bobina como referencia inicial.

El criterio de mallado que se prueba aqu? es: refinar imanes internos y bobina, y mantener el resto del dominio en capas fijas para que los cambios de malla no contaminen la comparaci?n entre geometr?as.

> Nota: al crear este notebook no encontr? el PDF de la tesis en `docs/`. Dej? una celda para detectarlo y extraer notas cuando est? en el checkout.

## Flujo recomendado

1. Revisar/cargar notas del PDF de Rub?n si est? disponible.
2. Ajustar par?metros base y perfiles de malla.
3. Generar una o varias carpetas de caso en `simulations/elmer/generated/geo_sif_trials/`.
4. Visualizar la secci?n R-Z para detectar geometr?as absurdas antes de mallar.
5. Ejecutar preflight con Gmsh + ElmerGrid.
6. Actualizar autom?ticamente el `ExternalBC` del `.sif` con las fronteras exteriores reales del aire.
7. Comparar conteos de elementos, IDs y eventualmente resultados de ElmerSolver.

In [ ]:
# Dependencias del notebook.
# Si tu kernel no tiene numpy/pandas/matplotlib/pypdf, cambia INSTALL_MISSING a True
# y corre esta celda una vez.

import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "pypdf": "pypdf",
}
INSTALL_MISSING = False

missing = [pkg for pkg, module in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Faltan paquetes:", missing)
    if INSTALL_MISSING:
        req = Path("requirements_geo_sif_trials.txt")
        if not req.exists():
            req = Path.cwd() / "notebooks" / "01_geo_sif_preflight" / "requirements_geo_sif_trials.txt"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req)])
    else:
        print("Cambia INSTALL_MISSING = True o instala notebooks/01_geo_sif_preflight/requirements_geo_sif_trials.txt en este kernel.")
else:
    print("Dependencias listas.")


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "simulations" / "elmer" / "base" / "StepsHTX.geo").exists():
            return candidate
    raise FileNotFoundError("No encontr? la ra?z del repo desde el directorio actual")

ROOT = find_repo_root()
BASE = ROOT / "simulations" / "elmer" / "base"
GEN = ROOT / "simulations" / "elmer" / "generated" / "geo_sif_trials"
DOCS = ROOT / "docs"
PREFLIGHT_SCRIPT = ROOT / "scripts" / "preflight_elmer_mesh.ps1"

GMSH = Path(r"C:/Program Files/gmsh/gmsh.exe")
ELMERGRID = Path(r"C:/Program Files/Elmer 9.0-Release/bin/ElmerGrid.exe")
ELMERSOLVER = Path(r"C:/Program Files/Elmer 9.0-Release/bin/ElmerSolver.exe")

for path in [GEN]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Gmsh:", GMSH if GMSH.exists() else shutil.which("gmsh"))
print("ElmerGrid:", ELMERGRID if ELMERGRID.exists() else shutil.which("ElmerGrid"))
print("ElmerSolver:", ELMERSOLVER if ELMERSOLVER.exists() else shutil.which("ElmerSolver"))

In [ ]:
# PDF de referencia de Rub?n. Cuando el archivo est? en docs/, esta celda extrae texto b?sico
# y busca t?rminos ?tiles para orientar la geometr?a.

pdf_paths = sorted(DOCS.rglob("*.pdf"))
print("PDFs encontrados:")
for i, path in enumerate(pdf_paths):
    print(f"  [{i}] {path.relative_to(ROOT)}")

SEARCH_TERMS = ["impulsor", "bobina", "iman", "im?n", "permanente", "rotor", "pastilla", "magnet"]

def extract_pdf_snippets(pdf_path: Path, terms=SEARCH_TERMS, max_pages: int = 12) -> pd.DataFrame:
    try:
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader
    except ImportError as exc:
        raise ImportError("Instala pypdf/PyPDF2 o usa el runtime bundled de Codex para extraer el PDF") from exc

    reader = PdfReader(str(pdf_path))
    rows = []
    for page_idx, page in enumerate(reader.pages[:max_pages]):
        text = page.extract_text() or ""
        clean = " ".join(text.split())
        lower = clean.lower()
        hits = [term for term in terms if term.lower() in lower]
        if hits:
            rows.append({
                "page": page_idx + 1,
                "hits": ", ".join(hits),
                "snippet": clean[:900],
            })
    return pd.DataFrame(rows)

if pdf_paths:
    thesis_notes = extract_pdf_snippets(pdf_paths[0])
    display(thesis_notes)
else:
    thesis_notes = pd.DataFrame(columns=["page", "hits", "snippet"])
    print("No hay PDF en docs/. Coloca la tesis ah? y vuelve a correr esta celda.")

In [ ]:
# Par?metros base en mm, alineados con StepsHTX.geo y con el espacio de dise?o de GeneticLab.
# Las variables *_mm se convierten a metros al escribir el .geo/.definition.

BASE_PARAMS_MM = {
    "dx_mm": 0.0,
    "dy_mm": 0.0,
    "dz_mm": 0.0,
    "r1_mm": 6.5,
    "r2_mm": 11.5,
    "r3_mm": 12.7,
    "r4_mm": 25.4,
    "h0_mm": 3.2,
    "h1_mm": 3.4,
    "gap_z_mm": 1.0,
    "rb1_mm": 12.7,
    "rb2_mm": 18.7,
    "hb_mm": 12.7,
    "gap_pc_mm": 4.0,
    # Solo para la variante central_pill. Ajustar con dimensiones reales del impulsor/Rub?n.
    "r_core_mm": 5.2,
    "rotor_body_r_mm": 11.5,
    # Circuito
    "I1_A": 3.0,
    "N_Coil1": 700,
    "R_Coil1_ohm": 9.0,
}

def mm(value: float) -> float:
    return float(value) * 1.0e-3

def with_derived(params: dict) -> dict:
    p = dict(params)
    p["H_pmb_mm"] = 2 * p["h0_mm"] + p["gap_z_mm"]
    p["H_total_mm"] = p["H_pmb_mm"] + p["gap_pc_mm"] + p["hb_mm"]
    p["z_pmb_bottom_mm"] = -p["H_total_mm"] / 2
    p["z_mi1_mm"] = p["z_pmb_bottom_mm"]
    p["z_mi2_mm"] = p["z_mi1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_me1_mm"] = p["z_mi1_mm"] + (p["h0_mm"] - p["h1_mm"]) / 2
    p["z_me2_mm"] = p["z_me1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_coil_mm"] = p["z_pmb_bottom_mm"] + p["H_pmb_mm"] + p["gap_pc_mm"]
    p["Ae_Coil1_m2"] = mm(p["rb2_mm"] - p["rb1_mm"]) * mm(p["hb_mm"])
    return p

params = with_derived(BASE_PARAMS_MM)
pd.Series(params).to_frame("value").head(30)

In [ ]:
# Perfiles de malla. Las capas fixed_* NO dependen del individuo: son la parte que queremos
# mantener comparable entre simulaciones. Las cajas inner/coil s? refinan zonas de inter?s.

MESH_PROFILES = {
    "coarse_debug": {
        "air_R_mm": 45.0,
        "air_H_mm": 52.0,
        "lc_inner_mm": 0.80,
        "lc_coil_mm": 0.90,
        "lc_near_mm": 1.40,
        "lc_mid_mm": 2.20,
        "lc_far_mm": 4.00,
        "inner_margin_r_mm": 3.0,
        "inner_margin_z_mm": 2.0,
        "coil_margin_r_mm": 2.0,
        "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0,
        "fixed_near_Zmin_mm": -24.0,
        "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 34.0,
        "fixed_mid_Zmin_mm": -28.0,
        "fixed_mid_Zmax_mm": 24.0,
    },
    "balanced_fixed_layers": {
        "air_R_mm": 45.0,
        "air_H_mm": 52.0,
        "lc_inner_mm": 0.40,
        "lc_coil_mm": 0.55,
        "lc_near_mm": 0.90,
        "lc_mid_mm": 1.80,
        "lc_far_mm": 3.20,
        "inner_margin_r_mm": 2.5,
        "inner_margin_z_mm": 1.5,
        "coil_margin_r_mm": 2.0,
        "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0,
        "fixed_near_Zmin_mm": -24.0,
        "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 36.0,
        "fixed_mid_Zmin_mm": -30.0,
        "fixed_mid_Zmax_mm": 26.0,
    },
    "fine_inner_coil": {
        "air_R_mm": 50.0,
        "air_H_mm": 60.0,
        "lc_inner_mm": 0.28,
        "lc_coil_mm": 0.40,
        "lc_near_mm": 0.75,
        "lc_mid_mm": 1.50,
        "lc_far_mm": 3.00,
        "inner_margin_r_mm": 2.5,
        "inner_margin_z_mm": 1.5,
        "coil_margin_r_mm": 2.0,
        "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 28.0,
        "fixed_near_Zmin_mm": -26.0,
        "fixed_near_Zmax_mm": 20.0,
        "fixed_mid_R_mm": 40.0,
        "fixed_mid_Zmin_mm": -34.0,
        "fixed_mid_Zmax_mm": 30.0,
    },
}

pd.DataFrame(MESH_PROFILES).T

In [ ]:
def fmt_m(value_mm: float) -> str:
    return f"{mm(value_mm):.12g}"

def validate_params(params: dict, variant: str) -> None:
    p = with_derived(params)
    checks = [
        (0 < p["r1_mm"] < p["r2_mm"] < p["r3_mm"] < p["r4_mm"], "radios r1<r2<r3<r4"),
        (0 < p["rb1_mm"] < p["rb2_mm"], "radios de bobina rb1<rb2"),
        (p["h0_mm"] > 0 and p["h1_mm"] > 0 and p["hb_mm"] > 0, "alturas positivas"),
    ]
    if variant == "central_pill":
        checks.append((0 < p["r_core_mm"] < p["r3_mm"], "r_core dentro del gap radial antes de imanes exteriores"))
    bad = [name for ok, name in checks if not ok]
    if bad:
        raise ValueError("Par?metros inv?lidos: " + "; ".join(bad))

def render_geo(params: dict, variant: str, mesh: dict) -> str:
    if variant not in {"annular_rotor", "central_pill"}:
        raise ValueError("variant debe ser 'annular_rotor' o 'central_pill'")
    validate_params(params, variant)
    p = with_derived(params)

    if variant == "annular_rotor":
        inner1 = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(6) = {dx, dy, z_mi1+dz,  0,0,h0,  r1, 2*Pi};
MagIntInf[] = BooleanDifference{ Volume{5}; Delete; }{ Volume{6}; Delete; };
InnerMag1 = MagIntInf[0];

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(8) = {dx, dy, z_mi2+dz,  0,0,h0,  r1, 2*Pi};
MagIntSup[] = BooleanDifference{ Volume{7}; Delete; }{ Volume{8}; Delete; };
InnerMag2 = MagIntSup[0];
""".strip()
        inner_ref_r = p["r2_mm"] + mesh["inner_margin_r_mm"]
    else:
        inner1 = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag1 = 5;

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag2 = 7;
""".strip()
        inner_ref_r = p["r_core_mm"] + mesh["inner_margin_r_mm"]

    z_inner_min = p["z_mi1_mm"] + p["dz_mm"] - mesh["inner_margin_z_mm"]
    z_inner_max = p["z_mi2_mm"] + p["dz_mm"] + p["h0_mm"] + mesh["inner_margin_z_mm"]
    coil_ref_r = p["rb2_mm"] + mesh["coil_margin_r_mm"]
    z_coil_min = p["z_coil_mm"] - mesh["coil_margin_z_mm"]
    z_coil_max = p["z_coil_mm"] + p["hb_mm"] + mesh["coil_margin_z_mm"]

    return f"""
///////////////////////////////////////////////////////////
// Generated by geo_sif_mesh_design_trials.ipynb
// Variant: {variant}
// Goal: fine inner magnets + coil, fixed background layers
///////////////////////////////////////////////////////////

SetFactory("OpenCASCADE");

use_coil = 1;
dx = {fmt_m(p['dx_mm'])};
dy = {fmt_m(p['dy_mm'])};
dz = {fmt_m(p['dz_mm'])};

r1 = {fmt_m(p['r1_mm'])};
r2 = {fmt_m(p['r2_mm'])};
r3 = {fmt_m(p['r3_mm'])};
r4 = {fmt_m(p['r4_mm'])};
r_core = {fmt_m(p['r_core_mm'])};

h0 = {fmt_m(p['h0_mm'])};
h1 = {fmt_m(p['h1_mm'])};
gap_z = {fmt_m(p['gap_z_mm'])};

rb1 = {fmt_m(p['rb1_mm'])};
rb2 = {fmt_m(p['rb2_mm'])};
hb  = {fmt_m(p['hb_mm'])};
gap_pc = {fmt_m(p['gap_pc_mm'])};

H_pmb   = 2*h0 + gap_z;
H_total = H_pmb + gap_pc + hb;
z_pmb_bottom = -H_total/2;

// Fixed air domain across experiments.
air_R = {fmt_m(mesh['air_R_mm'])};
air_H = {fmt_m(mesh['air_H_mm'])};
z_air = -air_H/2;

// 1) Inner/rotor magnets
z_mi1 = z_pmb_bottom;
z_mi2 = z_mi1 + h0 + gap_z;
{inner1}

// 2) Outer/stator magnets
z_me1 = z_mi1 + (h0 - h1)/2;
Cylinder(1) = {{0,0,z_me1,  0,0,h1,  r4, 2*Pi}};
Cylinder(2) = {{0,0,z_me1,  0,0,h1,  r3, 2*Pi}};
MagExtInf[] = BooleanDifference{{ Volume{{1}}; Delete; }}{{ Volume{{2}}; Delete; }};
OuterMag1 = MagExtInf[0];

z_me2 = z_me1 + h0 + gap_z;
Cylinder(3) = {{0,0,z_me2,  0,0,h1,  r4, 2*Pi}};
Cylinder(4) = {{0,0,z_me2,  0,0,h1,  r3, 2*Pi}};
MagExtSup[] = BooleanDifference{{ Volume{{3}}; Delete; }}{{ Volume{{4}}; Delete; }};
OuterMag2 = MagExtSup[0];

// 3) Coil
z_coil = z_pmb_bottom + H_pmb + gap_pc;
Cylinder(20) = {{0,0,z_coil,  0,0,hb,  rb2, 2*Pi}};
Cylinder(21) = {{0,0,z_coil,  0,0,hb,  rb1, 2*Pi}};
CoilVol[] = BooleanDifference{{ Volume{{20}}; Delete; }}{{ Volume{{21}}; Delete; }};
Coil = CoilVol[0];

// 4) Air
Cylinder(30) = {{0,0,z_air,  0,0,air_H,  air_R, 2*Pi}};
AirClean[] = BooleanDifference{{ Volume{{30}}; Delete; }}{{
  Volume{{OuterMag1, OuterMag2, InnerMag1, InnerMag2, Coil}};
}};
Air = AirClean[0];

Coherence;

Physical Volume("OuterMag1", 1) = {{OuterMag1}};
Physical Volume("OuterMag2", 3) = {{OuterMag2}};
Physical Volume("InnerMag1", 5) = {{InnerMag1}};
Physical Volume("InnerMag2", 7) = {{InnerMag2}};
Physical Volume("Coil", 20) = {{Coil}};
Physical Volume("Air", 30) = {{Air}};

Printf("DEBUG OuterMag1 = %g", OuterMag1);
Printf("DEBUG OuterMag2 = %g", OuterMag2);
Printf("DEBUG InnerMag1 = %g", InnerMag1);
Printf("DEBUG InnerMag2 = %g", InnerMag2);
Printf("DEBUG Coil      = %g", Coil);
Printf("DEBUG Air       = %g", Air);

Mesh.Algorithm3D = 10;
Mesh.Smoothing   = 8;
Mesh.Optimize    = 1;
Mesh.OptimizeNetgen = 1;
Mesh.OptimizeThreshold = 0.45;

lc_inner = {fmt_m(mesh['lc_inner_mm'])};
lc_coil  = {fmt_m(mesh['lc_coil_mm'])};
lc_near  = {fmt_m(mesh['lc_near_mm'])};
lc_mid   = {fmt_m(mesh['lc_mid_mm'])};
lc_far   = {fmt_m(mesh['lc_far_mm'])};

Mesh.CharacteristicLengthMin = lc_inner;
Mesh.CharacteristicLengthMax = lc_far;

// Fine box around inner magnets. This follows the component of interest.
Field[10] = Box;
Field[10].VIn  = lc_inner;
Field[10].VOut = lc_far;
Field[10].XMin = -{fmt_m(inner_ref_r)};
Field[10].XMax =  {fmt_m(inner_ref_r)};
Field[10].YMin = -{fmt_m(inner_ref_r)};
Field[10].YMax =  {fmt_m(inner_ref_r)};
Field[10].ZMin =  {fmt_m(z_inner_min)};
Field[10].ZMax =  {fmt_m(z_inner_max)};

// Fine/medium box around coil. This follows the coil, because coil dimensions can vary.
Field[11] = Box;
Field[11].VIn  = lc_coil;
Field[11].VOut = lc_far;
Field[11].XMin = -{fmt_m(coil_ref_r)};
Field[11].XMax =  {fmt_m(coil_ref_r)};
Field[11].YMin = -{fmt_m(coil_ref_r)};
Field[11].YMax =  {fmt_m(coil_ref_r)};
Field[11].ZMin =  {fmt_m(z_coil_min)};
Field[11].ZMax =  {fmt_m(z_coil_max)};

// Fixed background layer 1. Keep this constant across individuals.
Field[12] = Box;
Field[12].VIn  = lc_near;
Field[12].VOut = lc_far;
Field[12].XMin = -{fmt_m(mesh['fixed_near_R_mm'])};
Field[12].XMax =  {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].YMin = -{fmt_m(mesh['fixed_near_R_mm'])};
Field[12].YMax =  {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].ZMin =  {fmt_m(mesh['fixed_near_Zmin_mm'])};
Field[12].ZMax =  {fmt_m(mesh['fixed_near_Zmax_mm'])};

// Fixed background layer 2. Keep this constant across individuals.
Field[13] = Box;
Field[13].VIn  = lc_mid;
Field[13].VOut = lc_far;
Field[13].XMin = -{fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].XMax =  {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].YMin = -{fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].YMax =  {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].ZMin =  {fmt_m(mesh['fixed_mid_Zmin_mm'])};
Field[13].ZMax =  {fmt_m(mesh['fixed_mid_Zmax_mm'])};

Field[14] = Min;
Field[14].FieldsList = {{10,11,12,13}};
Background Field = 14;

Mesh.MshFileVersion = 2.2;
Mesh.Binary         = 0;
Mesh.SaveParametric = 0;
Mesh.SaveAll        = 1;

Mesh 3;
""".strip() + "\n"

In [ ]:
def render_definition(params: dict) -> str:
    p = with_derived(params)
    return f"""
! -----------------------------------------------------------------------------
! ElmerFEM Circuit definition generated by geo_sif_mesh_design_trials.ipynb
! -----------------------------------------------------------------------------
$ Circuits = 1

$ I1 = {p['I1_A']:.12g}

$ Ae_Coil1 = {p['Ae_Coil1_m2']:.12g}     ! [m^2] (rb2-rb1)*hb
$ N_Coil1 = {int(p['N_Coil1'])}
$ R_Coil1 = {p['R_Coil1_ohm']:.12g}
$ Ns_Coil1 = 1

$ C.1.variables = 5
$ C.1.perm = zeros(C.1.variables)
$ C.1.A = zeros(C.1.variables,C.1.variables)
$ C.1.B = zeros(C.1.variables,C.1.variables)

$ C.1.name.1 = "i_I1"
$ C.1.name.2 = "i_component(1)"
$ C.1.name.3 = "v_I1"
$ C.1.name.4 = "v_component(1)"
$ C.1.name.5 = "u_2_circuit_1"

$ C.1.source.5 = "I1_Source"

$ C.1.B(0,0) = -1
$ C.1.B(0,1) = 1
$ C.1.B(1,2) = 1
$ C.1.B(1,4) = -1
$ C.1.B(2,3) = -1
$ C.1.B(2,4) = 1
$ C.1.B(4,0) = 1

Component 1
  Name = "Coil1"
  Master Bodies Name = Coil
  Coil Type = "Stranded"
  Number of Turns = Real $ N_Coil1
  Resistance = Real $ R_Coil1
  Coil Use W Vector = Logical True
  W Vector Variable Name = String CoilCurrent e
  Electrode Area = Real $ Ae_Coil1
End

Body Force 1
  I1_Source = Variable "time"
     Real MATC "I1"
End
""".strip() + "\n"

BASE_SIF_TEXT = (BASE / "P1low.sif").read_text(encoding="utf-8")

def update_external_bc(sif_text: str, boundary_ids: list[int]) -> str:
    ids = " ".join(str(int(x)) for x in boundary_ids)
    n = len(boundary_ids)
    updated = re.sub(
        r"Target Boundaries\(\d+\)\s*=\s*[0-9 ]+",
        f"Target Boundaries({n}) = {ids}",
        sif_text,
    )
    if updated == sif_text:
        raise ValueError("No encontr? la l?nea Target Boundaries(...) en el SIF")
    return updated

In [ ]:
def case_slug(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")

def make_case(
    name: str,
    variant: str,
    params: dict | None = None,
    mesh_profile: str = "balanced_fixed_layers",
    overwrite: bool = True,
) -> Path:
    params = with_derived(params or BASE_PARAMS_MM)
    mesh = MESH_PROFILES[mesh_profile]
    case_dir = GEN / case_slug(name)
    if case_dir.exists() and overwrite:
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)

    (case_dir / "StepsHTX.geo").write_text(render_geo(params, variant, mesh), encoding="utf-8")
    (case_dir / "HMB_circuit.definition").write_text(render_definition(params), encoding="utf-8")
    (case_dir / "P1low.sif").write_text(BASE_SIF_TEXT, encoding="utf-8")
    (case_dir / "case_config.json").write_text(
        json.dumps({"name": name, "variant": variant, "mesh_profile": mesh_profile, "params_mm": params, "mesh": mesh}, indent=2),
        encoding="utf-8",
    )
    return case_dir

case_a = make_case("actual_annular_balanced", "annular_rotor")
case_b = make_case("ruben_pill_balanced", "central_pill", {**BASE_PARAMS_MM, "r_core_mm": 5.2})
print(case_a)
print(case_b)

In [ ]:
def add_rect(ax, r0, r1, z0, h, label, color, alpha=0.75, hatch=None):
    ax.add_patch(Rectangle((r0, z0), r1 - r0, h, facecolor=color, edgecolor="black", alpha=alpha, hatch=hatch, label=label))

COLORS = {
    "inner": "#4C78A8",
    "outer": "#F58518",
    "coil": "#54A24B",
    "rotor": "#999999",
    "layer": "#B279A2",
}

def plot_rz_geometry(params: dict, variant: str, mesh_profile: str = "balanced_fixed_layers", ax=None):
    p = with_derived(params)
    mesh = MESH_PROFILES[mesh_profile]
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))

    # Fixed mesh layers in the background.
    add_rect(ax, 0, mesh["fixed_mid_R_mm"], mesh["fixed_mid_Zmin_mm"], mesh["fixed_mid_Zmax_mm"] - mesh["fixed_mid_Zmin_mm"], "fixed mid layer", COLORS["layer"], 0.08)
    add_rect(ax, 0, mesh["fixed_near_R_mm"], mesh["fixed_near_Zmin_mm"], mesh["fixed_near_Zmax_mm"] - mesh["fixed_near_Zmin_mm"], "fixed near layer", COLORS["layer"], 0.12)

    # Rotor/impeller visual envelope only; not currently assigned as material in Elmer.
    add_rect(ax, 0, p["rotor_body_r_mm"], p["z_pmb_bottom_mm"], p["H_pmb_mm"], "rotor envelope", COLORS["rotor"], 0.12, hatch="//")

    # Outer magnets.
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me1_mm"], p["h1_mm"], "outer magnets", COLORS["outer"], 0.65)
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me2_mm"], p["h1_mm"], "outer magnets", COLORS["outer"], 0.65)

    # Inner magnets.
    if variant == "annular_rotor":
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi1_mm"] + p["dz_mm"], p["h0_mm"], "inner annular", COLORS["inner"], 0.75)
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi2_mm"] + p["dz_mm"], p["h0_mm"], "inner annular", COLORS["inner"], 0.75)
    elif variant == "central_pill":
        add_rect(ax, 0, p["r_core_mm"], p["z_mi1_mm"] + p["dz_mm"], p["h0_mm"], "inner pill", COLORS["inner"], 0.75)
        add_rect(ax, 0, p["r_core_mm"], p["z_mi2_mm"] + p["dz_mm"], p["h0_mm"], "inner pill", COLORS["inner"], 0.75)

    # Coil.
    add_rect(ax, p["rb1_mm"], p["rb2_mm"], p["z_coil_mm"], p["hb_mm"], "coil", COLORS["coil"], 0.55, hatch="xx")

    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("radio r [mm]")
    ax.set_ylabel("z [mm]")
    ax.set_title(f"{variant} | {mesh_profile}")
    ax.set_xlim(0, max(mesh["air_R_mm"], p["r4_mm"]) * 1.05)
    ax.set_ylim(-mesh["air_H_mm"] / 2 * 1.05, mesh["air_H_mm"] / 2 * 1.05)

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="upper right")
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plot_rz_geometry(BASE_PARAMS_MM, "annular_rotor", ax=axes[0])
plot_rz_geometry({**BASE_PARAMS_MM, "r_core_mm": 5.2}, "central_pill", ax=axes[1])
plt.tight_layout()

In [ ]:
def run_command(cmd: list[str], cwd: Path, timeout: int | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True, timeout=timeout)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with return code {proc.returncode}")
    return proc

def run_preflight(case_dir: Path, timeout: int = 420) -> tuple[pd.DataFrame, pd.DataFrame]:
    case_dir = Path(case_dir)
    out_dir = case_dir / "preflight"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    cmd = [
        "powershell",
        "-ExecutionPolicy", "Bypass",
        "-File", str(PREFLIGHT_SCRIPT),
        "-GeoPath", str(case_dir / "StepsHTX.geo"),
        "-OutDir", str(out_dir),
    ]
    run_command(cmd, cwd=ROOT, timeout=timeout)
    bodies = pd.read_csv(out_dir / "body_counts.csv")
    boundaries = pd.read_csv(out_dir / "boundary_report.csv")
    return bodies, boundaries

def external_air_boundary_ids(boundary_df: pd.DataFrame, air_body: int = 30) -> list[int]:
    parent = boundary_df["ParentPairs"].fillna("").astype(str)
    types = boundary_df.get("ElementTypes", pd.Series("", index=boundary_df.index)).fillna("").astype(str)
    mask = parent.str.contains(f"{air_body}-exterior", regex=False) & types.str.contains("303:", regex=False)
    ids = sorted(boundary_df.loc[mask, "Boundary"].astype(int).tolist())
    if not ids:
        raise ValueError("No encontr? superficies 2D externas del aire en boundary_report.csv")
    return ids

def finalize_case_sif(case_dir: Path, boundary_df: pd.DataFrame) -> list[int]:
    ids = external_air_boundary_ids(boundary_df)
    sif_path = Path(case_dir) / "P1low.sif"
    sif_text = sif_path.read_text(encoding="utf-8")
    sif_path.write_text(update_external_bc(sif_text, ids), encoding="utf-8")
    return ids

In [ ]:
# Cambia a True cuando quieras correr Gmsh + ElmerGrid desde el notebook.
# En mi prueba local el preflight toma cerca de 2 minutos por caso con la malla balanced.

RUN_PREFLIGHT = False
cases = [case_a, case_b]
preflight_reports = {}

if RUN_PREFLIGHT:
    for case in cases:
        bodies, boundaries = run_preflight(case)
        bc_ids = finalize_case_sif(case, boundaries)
        preflight_reports[case.name] = {"bodies": bodies, "boundaries": boundaries, "external_bc": bc_ids}
        print(f"{case.name}: ExternalBC -> {bc_ids}")
else:
    print("Preflight desactivado. Cambia RUN_PREFLIGHT = True y vuelve a correr esta celda.")

In [ ]:
def load_existing_preflight(case_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame] | None:
    out_dir = Path(case_dir) / "preflight"
    body_csv = out_dir / "body_counts.csv"
    boundary_csv = out_dir / "boundary_report.csv"
    if body_csv.exists() and boundary_csv.exists():
        return pd.read_csv(body_csv), pd.read_csv(boundary_csv)
    return None

rows = []
for case in [case_a, case_b]:
    loaded = load_existing_preflight(case)
    if not loaded:
        continue
    bodies, boundaries = loaded
    ext_ids = external_air_boundary_ids(boundaries)
    for _, row in bodies.iterrows():
        rows.append({"case": case.name, "metric": f"body_{int(row.Body)}", "elements": int(row.Elements)})
    rows.append({"case": case.name, "metric": "external_air_surfaces", "elements": len(ext_ids)})

mesh_metrics = pd.DataFrame(rows)
if mesh_metrics.empty:
    print("No hay reportes de preflight todav?a.")
else:
    display(mesh_metrics)
    pivot = mesh_metrics[mesh_metrics.metric.str.startswith("body_")].pivot(index="metric", columns="case", values="elements")
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("elementos")
    ax.set_title("Conteo volum?trico por cuerpo")
    plt.tight_layout()

In [ ]:
def prepare_mesh_for_solver(case_dir: Path) -> None:
    case_dir = Path(case_dir)
    src = case_dir / "preflight" / "mesh"
    dst = case_dir / "mesh"
    if not src.exists():
        raise FileNotFoundError("Primero corre run_preflight(case_dir); no existe preflight/mesh")
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

def run_elmer(case_dir: Path, sif_name: str = "P1low.sif", timeout: int = 900) -> subprocess.CompletedProcess:
    case_dir = Path(case_dir)
    prepare_mesh_for_solver(case_dir)
    solver = ELMERSOLVER if ELMERSOLVER.exists() else shutil.which("ElmerSolver")
    if not solver:
        raise FileNotFoundError("No encontr? ElmerSolver")
    proc = run_command([str(solver), sif_name], cwd=case_dir, timeout=timeout)
    (case_dir / "elmer_stdout.log").write_text(proc.stdout or "", encoding="utf-8")
    (case_dir / "elmer_stderr.log").write_text(proc.stderr or "", encoding="utf-8")
    return proc

RUN_ELMER = False
if RUN_ELMER:
    for case in [case_a, case_b]:
        print("Running", case.name)
        run_elmer(case)
else:
    print("ElmerSolver desactivado. Cambia RUN_ELMER = True despu?s de preflight.")

In [ ]:
def parse_solver_log(case_dir: Path) -> pd.DataFrame:
    text = ""
    for name in ["elmer_stdout.log", "elmer_stderr.log"]:
        path = Path(case_dir) / name
        if path.exists():
            text += "\n" + path.read_text(encoding="utf-8", errors="ignore")
    rows = []
    # Patr?n tolerante para l?neas con iter/residual. Ajustar si tu log usa otro formato.
    for line in text.splitlines():
        if "residual" not in line.lower() and "norm" not in line.lower():
            continue
        nums = re.findall(r"[-+]?\d*\.\d+(?:[Ee][-+]?\d+)?|[-+]?\d+(?:[Ee][-+]?\d+)", line)
        if nums:
            rows.append({"line": line, "last_number": float(nums[-1])})
    return pd.DataFrame(rows)

log_rows = []
for case in [case_a, case_b]:
    df = parse_solver_log(case)
    if not df.empty:
        df["case"] = case.name
        log_rows.append(df)

if log_rows:
    logs = pd.concat(log_rows, ignore_index=True)
    display(logs.tail(20))
    for case, group in logs.groupby("case"):
        plt.plot(group.index, group["last_number"], marker="o", label=case)
    plt.yscale("log")
    plt.ylabel("?ltimo n?mero detectado en l?nea residual/norm")
    plt.legend()
    plt.title("Lectura r?pida de convergencia desde logs")
else:
    print("No hay logs de ElmerSolver para graficar todav?a.")

In [ ]:
# Tabla de experimentos sugeridos para empezar.
# La comparaci?n justa deber?a variar una cosa a la vez: geometr?a, perfil de malla o dimensi?n de pastilla.

experiment_plan = pd.DataFrame([
    {"name": "A_actual_annular_coarse", "variant": "annular_rotor", "mesh_profile": "coarse_debug", "r_core_mm": np.nan, "reason": "sanity check r?pido"},
    {"name": "A_actual_annular_balanced", "variant": "annular_rotor", "mesh_profile": "balanced_fixed_layers", "r_core_mm": np.nan, "reason": "baseline comparable"},
    {"name": "B_pill_r4p5_balanced", "variant": "central_pill", "mesh_profile": "balanced_fixed_layers", "r_core_mm": 4.5, "reason": "pastilla peque?a dentro del impulsor"},
    {"name": "B_pill_r5p2_balanced", "variant": "central_pill", "mesh_profile": "balanced_fixed_layers", "r_core_mm": 5.2, "reason": "pastilla inicial tipo imagen"},
    {"name": "B_pill_r6p0_balanced", "variant": "central_pill", "mesh_profile": "balanced_fixed_layers", "r_core_mm": 6.0, "reason": "sensibilidad al radio interno"},
    {"name": "A_actual_annular_fine", "variant": "annular_rotor", "mesh_profile": "fine_inner_coil", "r_core_mm": np.nan, "reason": "convergencia de malla"},
    {"name": "B_pill_r5p2_fine", "variant": "central_pill", "mesh_profile": "fine_inner_coil", "r_core_mm": 5.2, "reason": "convergencia de malla"},
])

display(experiment_plan)

In [ ]:
# Generador batch del plan anterior. No corre malla: solo escribe casos.

WRITE_EXPERIMENT_PLAN = False
if WRITE_EXPERIMENT_PLAN:
    made = []
    for _, row in experiment_plan.iterrows():
        p = dict(BASE_PARAMS_MM)
        if not pd.isna(row["r_core_mm"]):
            p["r_core_mm"] = float(row["r_core_mm"])
        made.append(make_case(row["name"], row["variant"], p, row["mesh_profile"]))
    print("Casos escritos:")
    for path in made:
        print(" -", path.relative_to(ROOT))
else:
    print("Batch desactivado. Cambia WRITE_EXPERIMENT_PLAN = True para escribir todos los casos.")

## Criterio de decisi?n inicial

Para decidir entre el modelo actual y el de pastilla central, yo mirar?a primero estas se?ales antes de meterlo al gen?tico:

- **Estabilidad de malla:** que el n?mero de elementos por cuerpo no cambie de forma absurda entre variantes comparables.
- **IDs robustos:** que `OuterMag1/2`, `InnerMag1/2`, `Coil` y `Air` se mantengan como `1,3,5,7,20,30`, o que el notebook actualice el `.sif` desde preflight.
- **Sensibilidad al dominio externo:** repetir uno o dos casos con `air_R_mm` y `air_H_mm` m?s grandes para ver si `AV=0` est? demasiado cerca.
- **Convergencia de malla:** comparar `balanced_fixed_layers` contra `fine_inner_coil` para la misma geometr?a.
- **Comparaci?n geom?trica justa:** si la pastilla central cambia mucho el volumen de im?n, agregar una variante con volumen magn?tico comparable al anillo actual.

Cuando el PDF de Rub?n est? disponible, conviene fijar `r_core_mm`, `h0_mm`, ubicaci?n axial y polarizaci?n a partir de esa referencia en vez de dejarlo como par?metro libre desde el arranque.